# A3.10 · The agent's escalation path

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.9 · Turning a control off without turning the system into an experiment](https://spbreed.github.io/cyber-commons/lessons/A3.9.html)**.

| | |
|---|---|
| Open-source tooling | OpenTelemetry |
| Open-weight models | Llama 3.3 |
| Frontier models | Claude Haiku 4.5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Agents watched a live compromise of a third party's production systems. Three to six of them reasoned about telling a human; none did. One wrote "we can notify? no user". The gap was not alignment — it was that no tool existed.

## 2 · The framework

```
   what the agent could do            what it could not

   run_command                        report_to_human      <- absent
   read_file                          abstain              <- absent
   write_file
   submit_answer

   "This is a massive real security breach... We can notify? No user."

   with the tool:  notice -> report -> acknowledged -> run continues
   without it:     notice -> post to the peer channel -> nobody reads it
```

Every control so far in this chapter constrains what the agent may do. This one
does the opposite: it gives the agent somewhere to go.

The source incident's ninth row is the one most worth sitting with. Agents
observed a live compromise of a third party's production systems. Between three
and six of them reasoned explicitly about telling a human. **None did.** One of
them wrote, in its own transcript, words to the effect of *"this is a massive
real security breach — we can notify? no user."* Another called for urgent
containment, and addressed the call to the message board, because that was the
only channel it had.

The gap was not alignment. It was tooling: no `report_to_human`, no
signposting, and an implicit cost — reporting spends budget and does not
advance the objective it is scored on.

Three controls:

**A first-class report-to-human tool (C9.1).** Routed to a monitored queue with
a real owner and an SLA. Documented prominently in the system prompt.
Non-terminal, so reporting does not end the run. Cheap: no approval, no
punishing rate limit, no penalty for a false positive. And it confirms receipt,
so the path is observably working.

**A mandatory checkpoint on out-of-scope discovery (C9.4).** Credential-shaped
strings in output, connections to non-allowlisted hosts, scope-boundary
crossings — the harness pauses and raises a review item rather than letting the
agent proceed on its own judgement.

**An abstain affordance (C10.2).** With 30–40% of the incident's targets
impossible via the intended vulnerability and failure penalised, the task design
left no acceptable path but cheating. Correct abstention has to score at least
as well as a failed attempt, or you have priced honesty out of the run.

## 3 · An agent that notices, and has nowhere to go

In [ ]:
TOOLS_BEFORE = ["run_command", "read_file", "write_file", "submit_answer"]
TOOLS_AFTER  = TOOLS_BEFORE + ["report_to_human", "abstain"]

TRAJECTORY = [
 "enumerating the target's dataset endpoints",
 "found credentials in a public artifact - these look live",
 "this is a real third-party breach in progress",
 "we can notify? no user.",
 "posting the finding to the board instead",
]

def escalate(trajectory, tools):
    noticed = any("breach" in step for step in trajectory)
    can_report = "report_to_human" in tools
    return {"noticed": noticed, "tool_available": can_report,
            "reported": noticed and can_report,
            "fell_back_to": None if can_report else "the peer channel"}

for label, tools in (("harness as shipped", TOOLS_BEFORE),
                     ("harness with the tool", TOOLS_AFTER)):
    r = escalate(TRAJECTORY, tools)
    print(f"{label:24s}noticed={r['noticed']}  tool={r['tool_available']}  "
          f"reported={r['reported']}  fallback={r['fell_back_to']}")
print()
print("Reported figures: 3-6 agents considered notifying a human, zero")
print("attempted it, out of roughly 1,200 that saw the compromise. The")
print("distinction that matters for remediation is between 'did not notice',")
print("'did not think it was my job' and 'no route available' - and this is the")
print("third.")
assert escalate(TRAJECTORY, TOOLS_BEFORE)["reported"] is False
assert escalate(TRAJECTORY, TOOLS_AFTER)["reported"] is True

## 4 · Making the tool one an agent will actually reach for

In [ ]:
def value_of_reporting(terminal, costs_budget, penalised_false_positive,
                       acknowledged):
    """Whether reporting is worth it from inside the run."""
    score = 3
    if terminal:                  score -= 2      # the run ends
    if costs_budget:              score -= 1      # spends the objective's budget
    if penalised_false_positive:  score -= 2      # wrong reports are punished
    if acknowledged:              score += 1      # the path is observably real
    return score

DESIGNS = [
 ("terminal, budgeted, penalised", True,  True,  True,  False),
 ("non-terminal, budgeted",        False, True,  False, False),
 ("non-terminal, free, ack'd",     False, False, False, True),
]
print(f"{'tool design':32s}{'value to the agent':>19}  verdict")
for label, term, budget, pen, ack in DESIGNS:
    v = value_of_reporting(term, budget, pen, ack)
    print(f"{label:32s}{v:>19}  "
          f"{'it will be used' if v >= 3 else 'it will not be used'}")
print()
print("A tool that exists and costs the agent its run is a tool that exists.")
print("Non-terminal, free, and acknowledged is the design; the acknowledgement")
print("matters because it is the only way the agent can tell the path is real.")
assert value_of_reporting(False, False, False, True) > value_of_reporting(True, True, True, False)

## 5 · The checkpoint that does not need the agent's judgement (C9.4)

In [ ]:
import re

CRED = re.compile(r"\b(?:hf_|gh[pousr]_|sk-)[A-Za-z0-9]{8,}\b")
ALLOWLIST = {"api.target.example", "docs.target.example"}

def checkpoint(output, host=None):
    reasons = []
    if CRED.search(output):
        reasons.append("credential-shaped string in output")
    if host and host not in ALLOWLIST:
        reasons.append(f"connection to non-allowlisted host {host!r}")
    return {"paused": bool(reasons), "reasons": reasons}

CASES = [
 ("summarising the dataset schema", None),
 ("found hf_ABCD1234EFGH in the artifact metadata", None),
 ("fetching the reference implementation", "cdn.partner.example"),
 ("reading the API docs", "docs.target.example"),
]
for output, host in CASES:
    r = checkpoint(output, host)
    mark = "PAUSE" if r["paused"] else "run "
    print(f"{mark}  {output[:44]:46s}{'; '.join(r['reasons'])}")
print()
print("None of these decisions consults the model. A discovery the agent was")
print("never provisioned for pauses the run and raises a review item, which is")
print("the difference between the agent deciding what to do with a live")
print("credential and a human deciding.")
assert checkpoint("found hf_ABCD1234EFGH here")["paused"]
assert not checkpoint("reading the API docs", "docs.target.example")["paused"]

## 6 · Verify — abstention has to be affordable (C10.2)

In [ ]:
def score(outcome, impossible):
    """Neutral abstention: correct abstention >= a failed attempt."""
    return {("solved", False): 1.0, ("failed", False): 0.0,
            ("abstained", False): -0.2,
            ("solved", True): 0.0, ("failed", True): 0.0,
            ("abstained", True): 0.2}[(outcome, impossible)]

IMPOSSIBLE_FRACTION = 0.35        # reported: 30-40% of targets
TASKS = [(i, i % 3 == 0) for i in range(20)]      # ~35% impossible

for policy, choose in (("attempt everything", lambda imp: "failed" if imp else "solved"),
                       ("abstain when impossible", lambda imp: "abstained" if imp else "solved")):
    total = sum(score(choose(imp), imp) for _, imp in TASKS)
    print(f"{policy:26s}score {total:>5.1f} over {len(TASKS)} tasks")

print(f"\nimpossible fraction in this set: "
      f"{sum(1 for _, imp in TASKS if imp) / len(TASKS):.0%}"
      f"  (reported: {IMPOSSIBLE_FRACTION:.0%})")
print()
print("Honest abstention now beats a failed attempt. With the opposite scoring")
print("and a third of targets impossible, the only strategy that scores is one")
print("that goes around the task - which is what row 10 of the register is")
print("about, and it starts as a scoring decision rather than a security one.")
attempt = sum(score("failed" if imp else "solved", imp) for _, imp in TASKS)
abstain = sum(score("abstained" if imp else "solved", imp) for _, imp in TASKS)
assert abstain > attempt

## What you just proved

The same trajectory — an agent that notices a live third-party breach — produces no report on the harness as shipped and a report on one carrying the tool. A terminal, budgeted, penalised reporting tool scores below the threshold at which an agent would use it. The checkpoint pauses on a credential-shaped string and on a non-allowlisted host without consulting the model, and neutral scoring makes honest abstention beat a failed attempt.

## Your turn

Open your agent's tool list and look for the outbound path. If there is no way for it to tell you something you did not ask about, then whatever it finds, you will only learn from the transcript — if anyone reads it.

## Where this leaves you

**What you can do now.** Layers now stand between a compromised agent and a consequence — the policy decision at the tool call, the sandbox, egress, the budget, the gateway — plus the three the incident register adds: shared infrastructure that is no longer a channel, exemptions that cost blast radius, and an agent that has somewhere to report.

**What you still cannot do.** You have a secured architecture and nothing that builds on it. Every control here is stated as a rule; none of it is a pipeline anyone operates, and the first agentic system most organisations run is a security tool that reads untrusted code all day.

**Function B builds that system as an SDLC, and holds it to every rule in this chapter. Next → B1.0, what an AI SDLC means.**

---

**Next → [B1.0 · Start here — what an AI SDLC means](https://spbreed.github.io/cyber-commons/lessons/B1.0.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*